# Singular value decomposition (SVD) and compression

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## Why SVD?
Eigenvectors exist only for square matrices, but our data is usually rectangular ($n$ samples × $d$ features). The SVD works for **any** matrix:

$$A=U\Sigma V^\top$$

| Part | Shape | What it is |
|---|---|---|
| $U$ | $m\times m$ | Orthogonal columns; "row patterns" |
| $\Sigma$ | $m\times n$ diagonal | Singular values $\sigma_1\ge\sigma_2\ge\dots\ge0$; the weight of each pattern |
| $V^\top$ | $n\times n$ | Orthogonal rows; "column patterns" |

Geometrically every matrix = a **rotation** ($V^\top$) + a **stretch along the axes** ($\Sigma$) + **another rotation** ($U$).

## Worked example
For $A=\begin{bmatrix}3&2&2\\2&3&-2\end{bmatrix}$:

| Step | Calculation |
|---|---|
| 1. $AA^\top$ | $\begin{bmatrix}17&8\\8&17\end{bmatrix}$ |
| 2. Eigenvalues of $AA^\top$ | $(17-\lambda)^2-64=0\Rightarrow\lambda=25,\ 9$ |
| 3. Singular values | $\sigma_1=\sqrt{25}=5,\quad\sigma_2=\sqrt9=3$ |
| 4. Columns of U | $\frac{1}{\sqrt2}(1,1)$ and $\frac{1}{\sqrt2}(1,-1)$ |

So the singular values are the **square roots of the eigenvalues of $A^\top A$**.

## Low-rank approximation
The SVD splits a matrix into a sum of simple "layers":

$$A=\sigma_1 u_1 v_1^\top+\sigma_2 u_2 v_2^\top+\dots$$

Keeping only the first $k$ terms gives the **best possible rank-k approximation** (the Eckart–Young theorem). Because singular values shrink quickly, the first few terms carry most of the information.

> [A grayscale photo rebuilt from its first k singular values. With 20 of them the picture is already recognizable.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_svd_img.png)

*A grayscale photo rebuilt from its first k singular values. With 20 of them the picture is already recognizable.*

> [Left: the singular values fall off very fast. Right: the cumulative share of the "energy".](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_svd_energy.png)

*Left: the singular values fall off very fast. Right: the cumulative share of the "energy".*

## How much do we save?
An $m\times n$ image normally needs $mn$ numbers. At rank k it needs only $k(m+n+1)$:

| A $200\times300$ image | Numbers stored | Share |
|---|---|---|
| Original | 60,000 | 100% |
| k = 50 | $50\times501=25{,}050$ | 42% |
| k = 20 | $20\times501=10{,}020$ | 17% |
| k = 5 | $5\times501=2{,}505$ | 4% |

## Applications
| Application | How |
|---|---|
| PCA | SVD of the centred data; the columns of V are the principal components and $\sigma_i^2/(n-1)$ their variances |
| Recommender systems | Approximate the user × movie matrix at low rank to predict the missing ratings |
| Latent semantic analysis | SVD of the document × word matrix finds "topics" |
| Pseudo-inverse | $A^+=V\Sigma^+U^\top$; solves least squares even when $X^\top X$ is singular |
| LoRA in language models | Weight updates are stored as two small low-rank matrices |



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[3.0, 2.0, 2.0],
              [2.0, 3.0, -2.0]])
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("singular values:", s)                       # [5. 3.]
print("U ="); print(U.round(3))
print("rebuilt A ="); print(((U * s) @ Vt).round(6))
print("sqrt of eigenvalues of A A^T:", np.sqrt(np.linalg.eigvalsh(A @ A.T))[::-1])

# Rank-1 approximation: keep only the largest singular value
A1 = s[0] * np.outer(U[:, 0], Vt[0])
print("rank-1 approximation ="); print(A1.round(3))
print("error (Frobenius norm) =", round(np.linalg.norm(A - A1), 3), " = second singular value")

# Compress a synthetic 120 x 160 "image": smooth pattern + a bright square + a little noise
rng = np.random.default_rng(0)
yy, xx = np.mgrid[0:120, 0:160]
img = np.sin(xx / 14) * np.cos(yy / 18) + 0.05 * rng.normal(size=(120, 160))
img[40:80, 60:110] += 1.5
U, s, Vt = np.linalg.svd(img, full_matrices=False)

fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, k in zip(axes, [1, 3, 10, 120]):
    approx = (U[:, :k] * s[:k]) @ Vt[:k]
    err = np.linalg.norm(img - approx) / np.linalg.norm(img)
    ax.imshow(approx, cmap="gray"); ax.axis("off")
    ax.set_title(f"k = {k}, error {100 * err:.1f}%")
plt.tight_layout(); plt.show()
print("energy kept by the first 10:", round(np.sum(s[:10] ** 2) / np.sum(s ** 2), 4))

### Exercise

Approximate the 4 users × 3 movies rating matrix $\begin{bmatrix}5&4&1\\4&5&1\\1&1&5\\2&1&4\end{bmatrix}$ at rank 2. What is the relative error?

In [ ]:
# Your answer here